### Import Dependencies

In [ ]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

### Mock Example

In [ ]:
prompt = """You are a helpful assistant.
Return an answer to the question
Question: what is your name?"""

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt},
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

In [ ]:
response

### Add Instructor (Structured Outputs)

In [ ]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

In [ ]:
response = client.create(
    messages=[
        {"role": "system" , "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [ ]:
response

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system" , "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [ ]:
response

In [ ]:
raw_response

In [ ]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer")
    answer: str = Field(description="Answer to the question")

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system" , "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [ ]:
response

### RAG Pipeline

In [ ]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, k=5):

    query_embbeding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embbeding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_contexts_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_contexts.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_contexts_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_contexts_ratings": retrieved_contexts_ratings
    }

def process_context(context):
    
    formated_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_contexts"], context["retrieved_contexts_ratings"]):
        formated_context += f"- ID: {id},  rating: {rating},  description: {chunk}\n"

    return formated_context


def build_prompt(preprocessed_context, question):
    prompt = f"""

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never user word context ad refer to ir as the available products.
- Do not use markdown formatting.

Context: 
{preprocessed_context}

Question:
{question}
"""
    
    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_contexts"]
    }

    return final_answer

In [ ]:
output = rag_pipeline("Any USB chargeble speaker?")

In [ ]:
output